In [1]:
import numpy as np
import math

from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import Matern, RBF, WhiteKernel, ConstantKernel
from sklearn.model_selection import KFold
from sklearn.metrics import mean_squared_error

# ============================================================
# WEEK 10 — FUNCTION 2 (TRANSPARENT GP-BASED LOCAL MAXIMISATION)
# What changed vs Week 9:
#  - Surrogate: Gaussian Process (interpretable mean + uncertainty)
#  - Acquisition: EI (primary) + UCB (exploration) with clear components
#  - Candidate generation: global + trust-region + boundary probes
#  - Extra transparency: prints top candidates and score breakdown
#  - Deterministic seeds for reproducibility
#  - Output x_next rounded to 6 decimals
# ============================================================

# ----------------------------
# 1) INPUT DATA (your history)
# ----------------------------
X = np.array([
    [0.66579958, 0.12396913],
    [0.87779099, 0.77862750],
    [0.14269907, 0.34900513],
    [0.84527543, 0.71112027],
    [0.45464714, 0.29045518],
    [0.57771284, 0.77197318],
    [0.43816606, 0.68501826],
    [0.34174959, 0.02869772],
    [0.33864816, 0.21386725],
    [0.70263656, 0.92656420],
    [0.92656400, 1.02656400],   # will be clipped to 1.0
    [0.74750400, 0.20897000],
    [0.68340600, 0.06376900],
    [0.58313700, 0.01254900],
    [0.99145700, 0.00174400],
    [0.99145700, 0.00174400],   # duplicate
    [0.99947700, 0.02152800],
    [0.00757800, 0.97735900],
    [1.00000000, 0.98231600]
], dtype=float)

y = np.array([
    0.53899612, 0.42058624, -0.06562362, 0.29399291, 0.21496451,
    0.02310555, 0.24461934, 0.03874902, -0.01385762, 0.61120522,
    -0.04199554, 0.28033031, 0.62973064, 0.06695726,
    0.11670354823827367, 0.11353912028668156, 0.02426575623315439,
    0.15170334833240093, 0.003163976344064868
], dtype=float)

# Bounds enforcement for black-box
X = np.clip(X, 0.0, 1.0)

# -------------------------------------------
# 2) DEDUPE (average y for identical points)
# -------------------------------------------
def dedupe_average(X, y):
    b = np.ascontiguousarray(X).view(
        np.dtype((np.void, X.dtype.itemsize * X.shape[1]))
    )
    _, inv = np.unique(b, return_inverse=True)
    Xu, yu = [], []
    for i in np.unique(inv):
        idx = np.where(inv == i)[0]
        Xu.append(X[idx[0]])
        yu.append(y[idx].mean())
    return np.array(Xu), np.array(yu)

X, y = dedupe_average(X, y)

# -------------------------------------------
# 3) GP model (interpretable uncertainty)
#    Kernel chosen for transparency:
#      Constant * Matern + White noise
# -------------------------------------------
def make_gp(seed=2026):
    # Matern is a common, interpretable smoothness prior for BO.
    kernel = ConstantKernel(1.0, (1e-2, 1e2)) * Matern(
        length_scale=np.ones(X.shape[1]),
        length_scale_bounds=(1e-2, 1e1),
        nu=2.5
    ) + WhiteKernel(noise_level=1e-5, noise_level_bounds=(1e-8, 1e-1))

    return GaussianProcessRegressor(
        kernel=kernel,
        normalize_y=True,
        n_restarts_optimizer=12,
        random_state=seed
    )

# Optional: quick CV sanity check (kept lightweight for transparency)
def gp_cv_mse(X, y, seed=2026):
    kf = KFold(n_splits=min(5, len(y)), shuffle=True, random_state=seed)
    mses = []
    for tr, te in kf.split(X):
        gp = make_gp(seed=seed)
        gp.fit(X[tr], y[tr])
        mu, _ = gp.predict(X[te], return_std=True)
        mses.append(mean_squared_error(y[te], mu))
    return float(np.mean(mses))

cv_mse = gp_cv_mse(X, y, seed=2026)

gp = make_gp(seed=2026)
gp.fit(X, y)

# -------------------------------------------
# 4) Acquisition functions (transparent)
# -------------------------------------------
def normal_pdf(z):
    return np.exp(-0.5 * z * z) / np.sqrt(2.0 * np.pi)

def normal_cdf(z):
    # CDF via erf
    return 0.5 * (1.0 + np.vectorize(math.erf)(z / np.sqrt(2.0)))

def expected_improvement(mu, std, y_best, xi=0.001):
    std = np.maximum(std, 1e-12)
    z = (mu - y_best - xi) / std
    return (mu - y_best - xi) * normal_cdf(z) + std * normal_pdf(z)

# -------------------------------------------
# 5) Candidate generation (global + local + edges)
# -------------------------------------------
rng = np.random.default_rng(2026)

idx_best = int(np.argmax(y))
x_best = X[idx_best]
y_best = float(y[idx_best])
n = len(y)

# Trust region radius shrinks as data grows (simple, explainable heuristic)
tr_sigma = float(np.clip(0.10 / np.sqrt(max(n, 1)), 0.02, 0.06))

N_global = 30000
N_local  = 30000
N_edge   = 8000

X_global = rng.uniform(0.0, 1.0, size=(N_global, 2))
X_local  = np.clip(rng.normal(loc=x_best, scale=tr_sigma, size=(N_local, 2)), 0.0, 1.0)

# Boundary probing: sometimes maxima sit on constraints
edges = rng.uniform(0.0, 1.0, size=(N_edge, 2))
mask = rng.random(N_edge) < 0.5
edges[mask, 0] = rng.choice([0.0, 1.0], size=mask.sum())
edges[~mask, 1] = rng.choice([0.0, 1.0], size=(~mask).sum())

Xcand = np.vstack([X_global, X_local, edges])

# -------------------------------------------
# 6) Score candidates with EI + UCB (interpretable mix)
# -------------------------------------------
mu, std = gp.predict(Xcand, return_std=True)

xi = 0.001
ei = expected_improvement(mu, std, y_best, xi=xi)

kappa = 2.0
ucb = mu + kappa * std

# Distance filter: avoid wasting a query extremely close to known points
dists = np.sqrt(((Xcand[:, None, :] - X[None, :, :]) ** 2).sum(axis=2))
min_dist = dists.min(axis=1)

min_sep = float(np.clip(0.10 / np.sqrt(max(n, 1)), 0.01, 0.03))
valid = min_dist >= min_sep

# Mild novelty penalty to reduce “sampling bias” collapse
# (still prioritises good EI/UCB, but slightly prefers less-explored areas)
novelty = min_dist  # higher = more novel
novelty = (novelty - novelty.mean()) / (novelty.std() + 1e-12)

# Normalise EI and UCB for stable weighting
def zscore(v):
    return (v - v.mean()) / (v.std() + 1e-12)

ei_z  = zscore(ei)
ucb_z = zscore(ucb)

# Week 10 transparent mix:
#  - EI dominates (exploit likely improvement over best)
#  - UCB keeps exploration alive (interpretable uncertainty use)
#  - Novelty term addresses dataset sampling gaps
score = 0.65 * ei_z + 0.25 * ucb_z + 0.10 * novelty

# Apply validity mask
score_masked = np.where(valid, score, -np.inf)
best_idx = int(np.argmax(score_masked))

x_next = np.round(Xcand[best_idx], 6)

# -------------------------------------------
# 7) Transparency prints (why this point?)
# -------------------------------------------
# Show top 5 candidates for auditability
topk = 5
top_idx = np.argsort(score_masked)[-topk:][::-1]

print("================================================")
print("WEEK 10 FUNCTION 2 — NEXT DATA POINT (<=6 DECIMALS)")
print("================================================")
print(f"GP kernel learned: {gp.kernel_}")
print(f"CV MSE (sanity check): {cv_mse:.6f}")
print(f"x_best = [{x_best[0]:.6f}, {x_best[1]:.6f}], y_best = {y_best:.6f}")
print(f"trust_sigma = {tr_sigma:.6f}, min_sep = {min_sep:.6f}")
print("------------------------------------------------")
print(f"x_next = [{x_next[0]:.6f}, {x_next[1]:.6f}]")
print(f"mu(x_next)   = {mu[best_idx]:.6f}")
print(f"std(x_next)  = {std[best_idx]:.6f}")
print(f"EI(x_next)   = {ei[best_idx]:.6f}")
print(f"UCB(x_next)  = {ucb[best_idx]:.6f}")
print(f"min_dist     = {min_dist[best_idx]:.6f}")
print(f"score_parts  = EI_w=0.65, UCB_w=0.25, nov_w=0.10")
print("------------------------------------------------")
print("Top candidates (for transparency):")
for rank, i in enumerate(top_idx, start=1):
    x = Xcand[i]
    print(
        f"{rank}) x=[{x[0]:.6f},{x[1]:.6f}]  "
        f"mu={mu[i]:.6f} std={std[i]:.6f} EI={ei[i]:.6f} "
        f"UCB={ucb[i]:.6f} min_dist={min_dist[i]:.6f} score={score_masked[i]:.6f}"
    )


C:\Anaconda3\Lib\site-packages\sklearn\gaussian_process\kernels.py:452: ConvergenceWarning: The optimal value found for dimension 1 of parameter k1__k2__length_scale is close to the specified upper bound 10.0. Increasing the bound and calling fit again may find a better value.
  warnings.warn(
C:\Anaconda3\Lib\site-packages\sklearn\gaussian_process\kernels.py:442: ConvergenceWarning: The optimal value found for dimension 0 of parameter k2__noise_level is close to the specified lower bound 1e-08. Decreasing the bound and calling fit again may find a better value.
  warnings.warn(
C:\Anaconda3\Lib\site-packages\sklearn\gaussian_process\kernels.py:452: ConvergenceWarning: The optimal value found for dimension 1 of parameter k1__k2__length_scale is close to the specified upper bound 10.0. Increasing the bound and calling fit again may find a better value.
  warnings.warn(
C:\Anaconda3\Lib\site-packages\sklearn\gaussian_process\kernels.py:452: ConvergenceWarning: The optimal value found for

WEEK 10 FUNCTION 2 — NEXT DATA POINT (<=6 DECIMALS)
GP kernel learned: 0.917**2 * Matern(length_scale=[0.0324, 10], nu=2.5) + WhiteKernel(noise_level=0.00694)
CV MSE (sanity check): 0.022766
x_best = [0.683406, 0.063769], y_best = 0.629731
trust_sigma = 0.023570, min_sep = 0.023570
------------------------------------------------
x_next = [0.691583, 0.489131]
mu(x_next)   = 0.638635
std(x_next)  = 0.033245
EI(x_next)   = 0.017588
UCB(x_next)  = 0.705125
min_dist     = 0.270001
score_parts  = EI_w=0.65, UCB_w=0.25, nov_w=0.10
------------------------------------------------
Top candidates (for transparency):
1) x=[0.691583,0.489131]  mu=0.638635 std=0.033245 EI=0.017588 UCB=0.705125 min_dist=0.270001 score=2.516990
2) x=[0.691372,0.505981]  mu=0.638708 std=0.033180 EI=0.017606 UCB=0.705067 min_dist=0.256453 score=2.502731
3) x=[0.692368,0.488980]  mu=0.638221 std=0.033532 EI=0.017455 UCB=0.705285 min_dist=0.269680 score=2.498725
4) x=[0.690733,0.502687]  mu=0.638810 std=0.032787 EI=0.01

C:\Anaconda3\Lib\site-packages\sklearn\gaussian_process\kernels.py:452: ConvergenceWarning: The optimal value found for dimension 1 of parameter k1__k2__length_scale is close to the specified upper bound 10.0. Increasing the bound and calling fit again may find a better value.
  warnings.warn(
